<a href="https://colab.research.google.com/github/eshaans1305/BIS_lab/blob/main/1BM24CS096_Lab4_Structural_Design_Opt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import math

nodes = np.array([
    [720.0, 360.0],
    [720.0, 0.0],
    [360.0, 360.0],
    [360.0, 0.0],
    [0.0, 360.0],
    [0.0, 0.0]
])

elements = np.array([
    [2, 0],
    [5, 3],
    [3, 1],
    [2, 3],
    [0, 1],
    [4, 2],
    [5, 2],
    [4, 3],
    [0, 3],
    [1, 2]
])

E = 1e7
rho = 0.1

n_nodes = len(nodes)
n_elem = len(elements)
dof_per_node = 2
n_dof = n_nodes * dof_per_node

loads = np.zeros(n_dof)
loads[2 * 1 + 1] = -1e5
loads[2 * 3 + 1] = -1e5

fixed_dofs = np.array([2*4, 2*4+1, 2*5, 2*5+1])
free_dofs = np.setdiff1d(np.arange(n_dof), fixed_dofs)

x_min = 0.1
x_max = 35.0
stress_limit = 25000.0
disp_limit = 2.0

def solve_truss(areas):
    K = np.zeros((n_dof, n_dof))
    lengths = np.zeros(n_elem)

    for i, elem in enumerate(elements):
        n1, n2 = elem
        x1, y1 = nodes[n1]
        x2, y2 = nodes[n2]

        L = np.hypot(x2 - x1, y2 - y1)
        lengths[i] = L

        cx = (x2 - x1) / L
        cy = (y2 - y1) / L

        T = np.array([cx, cy, -cx, -cy])
        k_local = (areas[i] * E / L) * np.outer(T, T)

        dofs = np.array([2*n1, 2*n1+1, 2*n2, 2*n2+1])
        for row in range(4):
            for col in range(4):
                K[dofs[row], dofs[col]] += k_local[row, col]

    U = np.zeros(n_dof)
    K_free = K[np.ix_(free_dofs, free_dofs)]
    R_free = loads[free_dofs]

    U[free_dofs] = np.linalg.solve(K_free, R_free)

    stresses = np.zeros(n_elem)
    for i, elem in enumerate(elements):
        n1, n2 = elem
        x1, y1 = nodes[n1]
        x2, y2 = nodes[n2]
        L = lengths[i]
        cx = (x2 - x1) / L
        cy = (y2 - y1) / L

        dofs = np.array([2*n1, 2*n1+1, 2*n2, 2*n2+1])
        u_elem = U[dofs]

        stresses[i] = (E / L) * np.dot(np.array([-cx, -cy, cx, cy]), u_elem)

    weight = np.sum(areas * lengths * rho)
    return weight, stresses, U

def objective_function(areas):
    weight, stresses, U = solve_truss(areas)

    penalty = 0.0
    for s in stresses:
        if abs(s) > stress_limit:
            penalty += (abs(s) - stress_limit) * 1e5

    for u in U:
        if abs(u) > disp_limit:
            penalty += (abs(u) - disp_limit) * 1e5

    return weight + penalty

def levy_flight(Lambda, size):
    sigma = (math.gamma(1 + Lambda) * np.sin(np.pi * Lambda / 2) /
             (math.gamma((1 + Lambda) / 2) * Lambda * 2**((Lambda - 1) / 2)))**(1 / Lambda)
    u = np.random.normal(0, sigma, size)
    v = np.random.normal(0, 1, size)
    step = u / (np.abs(v)**(1 / Lambda))
    return step

def cuckoo_search(pop_size=30, max_iter=200, pa=0.25):
    pop = np.random.uniform(x_min, x_max, (pop_size, n_elem))
    fitness = np.array([objective_function(ind) for ind in pop])

    best_idx = np.argmin(fitness)
    best_nest = pop[best_idx].copy()
    best_fitness = fitness[best_idx]

    for iteration in range(max_iter):
        for i in range(pop_size):
            step = levy_flight(1.5, n_elem)
            new_nest = pop[i] + 0.01 * step * (pop[i] - best_nest)
            new_nest = np.clip(new_nest, x_min, x_max)

            new_fit = objective_function(new_nest)
            if new_fit < fitness[i]:
                pop[i] = new_nest
                fitness[i] = new_fit

        for i in range(pop_size):
            if np.random.rand() < pa:
                j, k = np.random.choice(pop_size, 2, replace=False)
                step_size = np.random.rand() * (pop[j] - pop[k])
                pop[i] = pop[i] + step_size
                pop[i] = np.clip(pop[i], x_min, x_max)
                fitness[i] = objective_function(pop[i])

        current_best_idx = np.argmin(fitness)
        if fitness[current_best_idx] < best_fitness:
            best_fitness = fitness[current_best_idx]
            best_nest = pop[current_best_idx].copy()

    return best_nest, best_fitness

best_areas, best_weight = cuckoo_search(pop_size=40, max_iter=300, pa=0.25)
final_w, final_s, final_u = solve_truss(best_areas)

print(f"Optimized Weight: {final_w:.4f} lbs")
print("Optimized Cross-Sectional Areas (in^2):")
for idx, area in enumerate(best_areas):
    print(f" Member {idx+1}: {area:.4f}")

print("\nMaximum Member Stress Magnitude:", np.max(np.abs(final_s)))
print("Maximum Nodal Displacement Magnitude:", np.max(np.abs(final_u)))


Optimized Weight: 6598.7053 lbs
Optimized Cross-Sectional Areas (in^2):
 Member 1: 0.1953
 Member 2: 25.8475
 Member 3: 29.7189
 Member 4: 0.9651
 Member 5: 13.3758
 Member 6: 35.0000
 Member 7: 15.5517
 Member 8: 18.0295
 Member 9: 0.1000
 Member 10: 21.6108

Maximum Member Stress Magnitude: 9031.258876825907
Maximum Nodal Displacement Magnitude: 1.9291523471926413
